# Визуализация транзакционного графа клиентов

В этом ноутбуке мы визуализируем транзакционный граф клиентов, построенный с помощью Spark-скрипта.

In [ ]:
# Импортируем необходимые библиотеки
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import json
import os
import seaborn as sns
from pyvis.network import Network
import plotly.graph_objects as go
import plotly.express as px
from matplotlib.colors import rgb2hex
from IPython.display import HTML, display
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Загрузка графа и статистики
graph_path = "/home/jovyan/work/reports/client_graph.graphml"
stats_path = "/home/jovyan/work/reports/graph_stats.json"

# Загрузка графа
G = nx.read_graphml(graph_path)
print(f"Загружен граф с {G.number_of_nodes()} узлами и {G.number_of_edges()} рёбрами")

# Загрузка статистики
with open(stats_path, 'r') as f:
    stats = json.load(f)

# Вывод статистики
print("\nСтатистика графа:")
for key, value in stats.items():
    print(f"{key}: {value}")

# Преобразование атрибутов из строк в числа
for node, attr in G.nodes(data=True):
    G.nodes[node]['community_id'] = int(attr['community_id'])
    G.nodes[node]['influence_score'] = float(attr['influence_score'])
    G.nodes[node]['community_size'] = int(attr['community_size'])
    G.nodes[node]['w_degree'] = float(attr['w_degree'])
    G.nodes[node]['degree'] = int(attr['degree'])
    G.nodes[node]['is_client'] = str(node).startswith('client_')

for u, v, attr in G.edges(data=True):
    G.edges[u, v]['weight'] = float(attr['weight'])

# 2. Анализ сообществ

In [ ]:

# Получаем информацию о сообществах
communities = {}
for node, attr in G.nodes(data=True):
    comm_id = attr['community_id']
    if comm_id not in communities:
        communities[comm_id] = []
    communities[comm_id].append(node)

# Сортируем сообщества по размеру
sorted_communities = sorted(communities.items(), key=lambda x: len(x[1]), reverse=True)

# Выводим информацию о топ-10 сообществах
print(f"Всего сообществ: {len(communities)}")
print("\nТоп-10 сообществ по размеру:")
for i, (comm_id, nodes) in enumerate(sorted_communities[:10]):
    print(f"Сообщество {comm_id}: {len(nodes)} узлов")

# Создаем DataFrame с информацией о сообществах
community_data = []
for comm_id, nodes in communities.items():
    # Средний PageRank в сообществе
    avg_influence = np.mean([G.nodes[node]['influence_score'] for node in nodes])
    # Средняя взвешенная степень
    avg_w_degree = np.mean([G.nodes[node]['w_degree'] for node in nodes])
    # Количество внутренних рёбер
    internal_edges = sum(1 for u, v in G.edges() if u in nodes and v in nodes)
    
    community_data.append({
        'community_id': comm_id,
        'size': len(nodes),
        'avg_influence': avg_influence,
        'avg_w_degree': avg_w_degree,
        'internal_edges': internal_edges
    })

community_df = pd.DataFrame(community_data)
community_df = community_df.sort_values('size', ascending=False).reset_index(drop=True)

# Выводим таблицу с информацией о сообществах
display(community_df.head(10))

plt.figure(figsize=(16, 6))  # Увеличиваем ширину фигуры для двух графиков

# График 1: Плотность распределения размеров сообществ
plt.subplot(1, 2, 1)  # 1 строка, 2 столбца, позиция 1
sns.kdeplot(community_df['size'], fill=True, color='skyblue')
sns.rugplot(community_df['size'], height=0.05, color='navy')
plt.title('Плотность распределения размеров сообществ', pad=20)
plt.xlabel('Размер сообщества')
plt.ylabel('Плотность вероятности')

# График 2: Круговая диаграмма топ-5 сообществ
plt.subplot(1, 2, 2)  # 1 строка, 2 столбца, позиция 2

# Подготовка данных для круговой диаграммы
top5_sizes = community_df.head(5)['size'].values
other_size = community_df.iloc[5:]['size'].sum()
sizes = np.append(top5_sizes, other_size)
labels = [f'Сообщество {community_df.iloc[i]["community_id"]}\n({s} уз.)' for i, s in enumerate(top5_sizes)]
labels.append(f'Другие\n({other_size} уз.)')

# Цвета для диаграммы (берём из палитры seaborn)
colors = sns.color_palette('pastel')[0:6]

# Рисуем круговую диаграмму
wedges, texts, autotexts = plt.pie(
    sizes, 
    labels=labels, 
    autopct='%1.1f%%', 
    startangle=90,
    colors=colors,
    textprops={'fontsize': 9},
    wedgeprops={'edgecolor': 'white', 'linewidth': 1}
)

# Добавляем тень и заголовок
plt.setp(autotexts, size=10, weight='bold')
plt.title('Доля топ-5 сообществ в графе', pad=20)

# Выравниваем графики
plt.tight_layout(pad=3.0)
plt.show()



In [ ]:
# Функция для получения цветовой карты для сообществ
def get_community_colors(communities, cmap_name='tab20'):
    cmap = plt.cm.get_cmap(cmap_name, len(communities))
    return {comm_id: rgb2hex(cmap(i)[:3]) for i, comm_id in enumerate(communities.keys())}

# Получаем цвета для сообществ
community_colors = get_community_colors(communities)

def visualize_client_graph(G, weight_threshold=0.0, influence_scale=5):
    """
    Создает интерактивную визуализацию графа клиентов с настраиваемыми параметрами
    
    Параметры:
    - G: граф NetworkX
    - weight_threshold: минимальный вес ребра для отображения (по умолчанию 1.0)
    - influence_scale: масштаб для размера узлов по влиянию (по умолчанию 12)
    """
    pos = nx.spring_layout(G, seed=42, k=0.3)

    # Создаем edge traces только для рёбер с весом > weight_threshold
    edge_trace = go.Scatter(
        x=[],
        y=[],
        line=dict(width=0.5, color='#888'),
        hoverinfo='none',
        mode='lines')

    for u, v, attr in G.edges(data=True):
        if attr['weight'] > weight_threshold:  # Используем переданный параметр
            x0, y0 = pos[u]
            x1, y1 = pos[v]
            edge_trace['x'] += (x0, x1, None)
            edge_trace['y'] += (y0, y1, None)

    # Создаем node traces для каждого сообщества
    node_traces = {}
    for comm_id, color in community_colors.items():
        node_traces[comm_id] = go.Scatter(
            x=[],
            y=[],
            text=[],
            mode='markers',
            hoverinfo='text',
            marker=dict(
                color=color,
                size=[],
                line=dict(width=0.5, color='#888')
            ),
            name=f'Сообщество {comm_id}'
        )

    # Заполняем node traces
    for node, attr in G.nodes(data=True):
        x, y = pos[node]
        comm_id = attr['community_id']
        node_traces[comm_id]['x'] += (x,)
        node_traces[comm_id]['y'] += (y,)
        node_traces[comm_id]['text'] += (f"ID: {node}<br>Влияние: {attr['influence_score']:.4f}<br>Степень: {attr['degree']}",)
        node_traces[comm_id]['marker']['size'] += (attr['influence_score'] * influence_scale,)  # Используем переданный параметр

    # Создаем фигуру
    fig = go.Figure(
        data=[edge_trace] + list(node_traces.values()),
        layout=go.Layout(
            title=dict(
                text='Транзакционный граф клиентов',
                font=dict(size=16)
            ),
            showlegend=True,
            hovermode='closest',
            margin=dict(b=0, l=0, r=0, t=40),
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            height=800,
            legend=dict(x=1.05, y=0.5)
        )
    )

    # Сохраняем и отображаем
    return fig

# Отфильтрованный граф без слабых рёбер
fig = visualize_client_graph(G, weight_threshold=1.0, influence_scale=12)
fig.write_html('/home/jovyan/work/reports/client_graph_plotly_filtered.html')
fig.show()

# Полный граф
fig = visualize_client_graph(G)
fig.write_html('/home/jovyan/work/reports/client_graph_plotly_full.html')



In [ ]:
# Создаем интерактивную визуализацию с помощью PyVis
net = Network(height='800px', width='100%', notebook=True, bgcolor='#ffffff', font_color='#333333')

# Добавляем узлы
for node, attr in G.nodes(data=True):
    # Определяем размер узла на основе influence_score
    size = attr['influence_score'] * 50
    
    # Определяем цвет узла на основе сообщества
    color = community_colors[attr['community_id']]
    
    # Создаем метку узла
    title = f"ID: {node}<br>Сообщество: {attr['community_id']}<br>Влияние: {attr['influence_score']:.4f}<br>Степень: {attr['degree']}"  # Изменено с connections на degree

    
    # Добавляем узел в сеть
    net.add_node(node, title=title, size=size, color=color)

# Добавляем ребра
for u, v, attr in G.edges(data=True):
    # Определяем толщину ребра на основе веса
    width = attr['weight'] * 2
    
    # Создаем метку ребра
    title = f"Вес: {attr['weight']:.2f}<br>Типы: {attr['relationships']}"
    
    # Добавляем ребро в сеть
    net.add_edge(u, v, title=title, width=width, arrowStrikethrough=False)

# Настраиваем физику для лучшего отображения сообществ
net.barnes_hut(gravity=-10000, central_gravity=0.3, spring_length=200, spring_strength=0.05, damping=0.09)

# Сохраняем и отображаем
net.save_graph('/home/jovyan/work/reports/client_graph_pyvis.html')
# display(HTML('/home/jovyan/work/reports/client_graph_pyvis.html'))

In [ ]:
# Выбираем топ-5 сообществ для визуализации
top_communities = [comm_id for comm_id, _ in sorted_communities[:5]]
top_community_nodes = set()
for comm_id in top_communities:
    top_community_nodes.update(communities[comm_id])

# Создаем подграф только с узлами из топ-5 сообществ
top_subgraph = G.subgraph(top_community_nodes)
# print(f"Подграф с {top_subgraph.number_of_nodes()} узлами и {top_subgraph.number_of_edges()} рёбрами")

# # Визуализируем подграф с NetworkX
# plt.figure(figsize=(14, 12))

# # Используем spring_layout для размещения узлов
# pos_top = nx.spring_layout(top_subgraph, seed=42, k=0.7)

# # Рисуем узлы, цвет по сообществу, размер по influence_score
# node_sizes = [top_subgraph.nodes[node]['influence_score'] * 400 for node in top_subgraph.nodes()]
# node_colors = [community_colors[top_subgraph.nodes[node]['community_id']] for node in top_subgraph.nodes()]

# # Рисуем ребра, толщина по весу
# edge_weights = [top_subgraph.edges[u, v]['weight'] * 0.5 for u, v in top_subgraph.edges()]

# # Рисуем граф
# nx.draw_networkx_edges(top_subgraph, pos_top, width=edge_weights, alpha=0.3)
# nx.draw_networkx_nodes(top_subgraph, pos_top, node_size=node_sizes, node_color=node_colors, alpha=0.7)

# # Добавляем подписи для крупных сообществ
# for comm_id in top_communities:
#     nodes = communities[comm_id]
#     # Фильтруем только те узлы, которые есть в подграфе
#     nodes_in_subgraph = [node for node in nodes if node in top_subgraph.nodes()]
#     if nodes_in_subgraph:
#         # Находим центр сообщества
#         center_x = np.mean([pos_top[node][0] for node in nodes_in_subgraph])
#         center_y = np.mean([pos_top[node][1] for node in nodes_in_subgraph])
#         plt.text(center_x, center_y, f"C{comm_id}", fontsize=20, fontweight='bold', 
#                  ha='center', va='center', bbox=dict(facecolor='white', alpha=0.7))

# plt.title('Топ-5 сообществ транзакционного графа', fontsize=16)
# plt.axis('off')
# plt.tight_layout()
# plt.savefig('/home/jovyan/work/reports/client_graph_top5_communities.png', dpi=300)
# plt.show()

# Создаем интерактивную визуализацию подграфа с PyVis
net_top = Network(height='800px', width='100%', notebook=True, bgcolor='#ffffff', font_color='#333333')

# Добавляем узлы
for node, attr in top_subgraph.nodes(data=True):
    size = attr['influence_score'] * 50
    color = community_colors[attr['community_id']]
    title = f"ID: {node}<br>Сообщество: {attr['community_id']}<br>Влияние: {attr['influence_score']:.4f}<br>Степень: {attr['degree']}"
    net_top.add_node(node, title=title, size=size, color=color)

# Добавляем ребра
for u, v, attr in top_subgraph.edges(data=True):
    if attr['weight'] > 1.0:  # Фильтруем слабые рёбра
        width = attr['weight'] * 2
        title = f"Вес: {attr['weight']:.2f}<br>Типы: {attr['relationships']}"
        net_top.add_edge(u, v, title=title, width=width, arrowStrikethrough=False)

# Настраиваем физику для лучшего отображения сообществ
net_top.barnes_hut(gravity=-10000, central_gravity=0.3, spring_length=200, spring_strength=0.05, damping=0.09)

# Сохраняем и отображаем
net_top.save_graph('/home/jovyan/work/reports/client_graph_top5_pyvis.html')
# display(HTML('/home/jovyan/work/reports/client_graph_top5_pyvis.html'))

In [ ]:
# Создаем HTML-отчет с основными визуализациями
html_report = f"""
<html>
<head>
    <title>Отчет по анализу транзакционного графа</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 20px; }}
        h1, h2 {{ color: #333; }}
        .section {{ margin-bottom: 30px; }}
        .stats {{ border-collapse: collapse; width: 100%; }}
        .stats th, .stats td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
        .stats th {{ background-color: #f2f2f2; }}
        .image {{ margin-top: 20px; text-align: center; }}
        .image img {{ max-width: 100%; height: auto; }}
        .links {{ margin-top: 20px; }}
        .links a {{ display: inline-block; margin-right: 20px; padding: 10px; background-color: #4CAF50; color: white; text-decoration: none; border-radius: 5px; }}
        .links a:hover {{ background-color: #45a049; }}
    </style>
</head>
<body>
    <h1>Отчет по анализу транзакционного графа клиентов</h1>
    
    <div class="section">
        <h2>Статистика графа</h2>
        <table class="stats">
            <tr><th>Метрика</th><th>Значение</th></tr>
            <tr><td>Всего узлов</td><td>{stats['total_nodes']}</td></tr>
            <tr><td>Всего ребер</td><td>{stats['total_edges']}</td></tr>
            <tr><td>Отфильтровано узлов</td><td>{stats['filtered_nodes']}</td></tr>
            <tr><td>Отфильтровано ребер</td><td>{stats['filtered_edges']}</td></tr>
            <tr><td>Количество сообществ</td><td>{stats['communities']}</td></tr>
            <tr><td>Размер крупнейшего сообщества</td><td>{stats['largest_community_size']}</td></tr>
            <tr><td>Размер наименьшего сообщества</td><td>{stats['smallest_community_size']}</td></tr>
            <tr><td>Средний размер сообщества</td><td>{stats['avg_community_size']:.2f}</td></tr>
            <tr><td>Средняя степень</td><td>{stats['avg_degree']:.2f}</td></tr>
            <tr><td>Максимальная степень</td><td>{stats['max_degree']}</td></tr>
            <tr><td>Средняя взвешенная степень</td><td>{stats['avg_weighted_degree']:.2f}</td></tr>
        </table>
    </div>
    
    <div class="section">
        <h2>Визуализации графа</h2>
        <div class="image">
            <h3>Интерактивный граф (Plotly)</h3>
            <div class="plotly-container">
                <iframe src="client_graph_plotly_full.html" width="80%" height="1280px" frameborder="0"></iframe>
            </div>
        </div>

        <div class="image">
            <h3>Топ-5 сообществ (Plotly)</h3>
            <div class="plotly-container">
                <iframe src="client_graph_top5_pyvis.html" width="80%" height="1280px" frameborder="0"></iframe>
            </div>
        </div>
        
        
        <div class="links">
            <h3>Интерактивные визуализации</h3>
            <a href="client_graph_pyvis.html" target="_blank">PyVis (полный граф)</a>
            <a href="client_graph_top5_pyvis.html" target="_blank">PyVis (топ-5 сообществ)</a>
            <a href="client_graph_plotly_filtered.html" target="_blank">Plotly (отфильтрованный)</a>
        </div>
    </div>
</body>
</html>
"""

with open('/home/jovyan/work/reports/client_graph_communities.html', 'w') as f:
    f.write(html_report)

print("Визуализация сохранена в /home/jovyan/work/reports/client_graph_communities.html")